# Proposed pipeline (v2 — label-mapping fix)

Same TF-IDF + Chi2/MI feature selection + GridSearchCV pipeline as
`proposed_improvements.ipynb`, with the label-mapping bug fixed (I-1) and a corrected
evaluation: GridSearchCV now optimizes macro-F1 instead of positive-class-only F1, and
results report macro-F1, per-class precision/recall/F1, and a confusion matrix for every
model (I-2). `class_weight='balanced'` is added as a tunable option for LR/SVM (I-5).
Results are merged with the baseline + Dummy results from `baseline_pipeline_v2.ipynb`
into one comparable table (I-4). SHAP explainability is deferred to a later pass once the
corrected numbers are the ones being explained (see ISSUE_PLAN.md Phase 5).

**2026-08-24 update (CV-fold feature-selection leakage fix):** the Chi2/MI selectors
were previously fit once on the full training set (using `y_train`) *before*
`GridSearchCV`'s internal 5-fold CV ran -- a real, if narrow-impact, form of leakage
(selection saw labels of rows that later served as held-out CV folds during
hyperparameter search; this does not affect the test-set numbers, since selection was
never fit on valid/test, but it could bias which hyperparameters GridSearchCV picked as
"best"). Selection is now wrapped in an `sklearn.Pipeline` with the classifier so it is
refit inside every CV fold, matching the leak-free pattern `ablation_v1.ipynb` already
used.

**2026-08-24 update (train+valid merge):** `valid.csv` was previously loaded and scored
after model selection but never used for any decision -- a wasted split. It is now merged
into the training pool (`train_full = train + valid`) before TF-IDF fitting and
`GridSearchCV`, so tuning sees ~1,284 more labeled rows. Test stays untouched and is the
only held-out split reported. The DistilBERT reference notebook is deliberately *not*
changed to match -- it still follows the official split, so its training-data budget
differs from the classical models here; this is disclosed in the paper.

In [1]:
import re
import warnings

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report

Load data with the corrected label mapping

In [2]:
train_raw = load_and_label("train.csv")
valid_raw = load_and_label("valid.csv")
test = load_and_label("test.csv")

# Merge train+valid into one fitting pool (see the 2026-08-24 note above); test
# stays untouched and is the only held-out split reported below.
train = pd.concat([train_raw, valid_raw], ignore_index=True)

balance = train["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
print("Train(+valid) class balance:\n", balance)
assert 0.35 < balance["fake"] < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

Train(+valid) class balance:
 Label
real    0.557098
fake    0.442902
Name: proportion, dtype: float64


Text preprocessing (stopword removal + stemming)

In [3]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)


train["clean_text"] = train["Statement"].apply(preprocess)
test["clean_text"] = test["Statement"].apply(preprocess)

TF-IDF feature extraction

In [4]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.9)

X_train_tfidf = tfidf.fit_transform(train["clean_text"])
X_test_tfidf = tfidf.transform(test["clean_text"])

y_train = train["Label"]
y_test = test["Label"]

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (11524, 10000)


Chi-Square feature selection -- selector *factory* only. The selector itself is fit
inside each GridSearchCV/CV fold (see the Pipeline in the next code section), not here,
so it never sees labels from rows that later serve as a held-out CV fold.

In [5]:
k_features = 3000


def make_chi2_selector():
    return SelectKBest(score_func=chi2, k=k_features)


print(f"Chi-square selector factory ready (k={k_features}); fit happens per-CV-fold below.")

Chi-square selector factory ready (k=3000); fit happens per-CV-fold below.


Mutual Information feature selection -- selector factory only, same reasoning as the
Chi-square cell above: fit happens inside each CV fold, not on the full training set
beforehand.

In [6]:
def _mutual_info_score_func(X, y):
    # sklearn requires discrete_features=True/"auto" for sparse input (a dense
    # continuous-feature estimator isn't supported here), which routes through an
    # internal discrete-discrete MI helper that emits one benign UserWarning per
    # feature ("Clustering metrics expects discrete values..."). At 10,000 features x
    # every CV fold x every hyperparameter combination, that floods captured notebook
    # output to hundreds of MB. Suppressing it here (inside the function actually
    # executed by each GridSearchCV worker) is reliable across process boundaries in a
    # way that a notebook-level `warnings.filterwarnings` call is not; it changes no
    # computation, only what gets printed.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return mutual_info_classif(X, y, random_state=RANDOM_STATE)


def make_mi_selector():
    return SelectKBest(score_func=_mutual_info_score_func, k=k_features)


print(f"Mutual Information selector factory ready (k={k_features}); fit happens per-CV-fold below.")

Mutual Information selector factory ready (k=3000); fit happens per-CV-fold below.


GridSearch + evaluation -- macro-F1 is now the tuning objective (I-2). Feature selection
(chi2/MI) is wrapped in the same `sklearn.Pipeline` as the classifier and fit inside each
CV fold, so `GridSearchCV`'s internal cross-validation never sees a selector fit on labels
from its own held-out fold (matches the leak-free pattern `ablation_v1.ipynb` already
used for its Chi2 stage). Chi-square is cheap enough per fold that this refit adds
negligible runtime; Mutual Information is more expensive per fold but still tractable at
this dataset size without additional caching.

In [7]:
def train_and_evaluate(model, param_grid, selector_factory, X_train, y_train, X_test, y_test):
    pipe = Pipeline([("select", selector_factory()), ("clf", model)])
    prefixed_grid = {f"clf__{k}": v for k, v in param_grid.items()}
    grid = GridSearchCV(pipe, prefixed_grid, cv=5, scoring="f1_macro", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    test_metrics = evaluate_full(y_test, best_model.predict(X_test))

    return best_model, grid.best_params_, test_metrics

Model + param-grid definitions (class_weight='balanced' added to LR/SVM per I-5)

In [8]:
def make_models():
    return [
        (
            "Logistic Regression",
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "solver": ["liblinear"], "class_weight": [None, "balanced"]},
        ),
        (
            "SVM",
            LinearSVC(random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "class_weight": [None, "balanced"]},
        ),
        ("Naive Bayes", MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}),
        (
            "Random Forest",
            RandomForestClassifier(random_state=RANDOM_STATE),
            {
                "n_estimators": [100, 200],
                "max_depth": [None, 10],
                "min_samples_split": [2, 5],
            },
        ),
        (
            "XGBoost",
            XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
            {
                "n_estimators": [100, 200],
                "max_depth": [3, 6],
                "learning_rate": [0.01, 0.1],
            },
        ),
    ]

Run experiments (Chi² features)

In [9]:
results_chi2 = {}
best_models_chi2 = {}

for name, model, params in make_models():
    print(f"\nTraining {name} with Chi-square features (selected inside each CV fold)...")
    best_model, best_params, test_metrics = train_and_evaluate(
        model, params, make_chi2_selector, X_train_tfidf, y_train, X_test_tfidf, y_test
    )
    print("Best params:", best_params)
    print_report(name, y_test, best_model.predict(X_test_tfidf))
    results_chi2[name] = {"best_params": best_params, "test": test_metrics}
    best_models_chi2[name] = best_model


Training Logistic Regression with Chi-square features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'clf__solver': 'liblinear'}

Logistic Regression
[[340 213]
 [272 442]]
              precision    recall  f1-score   support

        fake      0.556     0.615     0.584       553
        real      0.675     0.619     0.646       714

    accuracy                          0.617      1267
   macro avg      0.615     0.617     0.615      1267
weighted avg      0.623     0.617     0.619      1267


Training SVM with Chi-square features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced'}

SVM
[[328 225]
 [261 453]]
              precision    recall  f1-score   support

        fake      0.557     0.593     0.574       553
        real      0.668     0.634     0.651       714

    accuracy                          0.616      1267
   macro avg      0.613     0.614     0.613      1267
weighted avg      0.620     0.616     0.618      1267


Training Naive Bayes with Chi-square features (selected inside each CV fold)...
Best params: {'clf__alpha': 0.5}

Naive Bayes
[[225 328]
 [167 547]]
              precision    recall  f1-score   support

        fake      0.574     0.407     0.476       553
        real      0.625     0.766     0.688       714

    accuracy                          0.609      1267
   macro avg      0.600     0.586     0.582      1267
weighted avg      0.603     0.609     0.596      1267


Training Random Forest with Chi-square features (selected inside each CV fold)...


Best params: {'clf__max_depth': None, 'clf__min_samples_split': 5, 'clf__n_estimators': 100}

Random Forest
[[266 287]
 [191 523]]
              precision    recall  f1-score   support

        fake      0.582     0.481     0.527       553
        real      0.646     0.732     0.686       714

    accuracy                          0.623      1267
   macro avg      0.614     0.607     0.607      1267
weighted avg      0.618     0.623     0.617      1267


Training XGBoost with Chi-square features (selected inside each CV fold)...


Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 6, 'clf__n_estimators': 200}

XGBoost
[[201 352]
 [125 589]]
              precision    recall  f1-score   support

        fake      0.617     0.363     0.457       553
        real      0.626     0.825     0.712       714

    accuracy                          0.624      1267
   macro avg      0.621     0.594     0.585      1267
weighted avg      0.622     0.624     0.601      1267



Run experiments (MI features)

In [10]:
results_mi = {}
best_models_mi = {}

for name, model, params in make_models():
    print(f"\nTraining {name} with MI features (selected inside each CV fold)...")
    best_model, best_params, test_metrics = train_and_evaluate(
        model, params, make_mi_selector, X_train_tfidf, y_train, X_test_tfidf, y_test
    )
    print("Best params:", best_params)
    print_report(name, y_test, best_model.predict(X_test_tfidf))
    results_mi[name] = {"best_params": best_params, "test": test_metrics}
    best_models_mi[name] = best_model


Training Logistic Regression with MI features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'clf__solver': 'liblinear'}

Logistic Regression
[[337 216]
 [261 453]]
              precision    recall  f1-score   support

        fake      0.564     0.609     0.586       553
        real      0.677     0.634     0.655       714

    accuracy                          0.624      1267
   macro avg      0.620     0.622     0.620      1267
weighted avg      0.628     0.624     0.625      1267


Training SVM with MI features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced'}

SVM
[[315 238]
 [260 454]]
              precision    recall  f1-score   support

        fake      0.548     0.570     0.559       553
        real      0.656     0.636     0.646       714

    accuracy                          0.607      1267
   macro avg      0.602     0.603     0.602      1267
weighted avg      0.609     0.607     0.608      1267


Training Naive Bayes with MI features (selected inside each CV fold)...


Best params: {'clf__alpha': 0.1}

Naive Bayes
[[230 323]
 [167 547]]
              precision    recall  f1-score   support

        fake      0.579     0.416     0.484       553
        real      0.629     0.766     0.691       714

    accuracy                          0.613      1267
   macro avg      0.604     0.591     0.587      1267
weighted avg      0.607     0.613     0.601      1267


Training Random Forest with MI features (selected inside each CV fold)...


Best params: {'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}

Random Forest
[[265 288]
 [206 508]]
              precision    recall  f1-score   support

        fake      0.563     0.479     0.518       553
        real      0.638     0.711     0.673       714

    accuracy                          0.610      1267
   macro avg      0.600     0.595     0.595      1267
weighted avg      0.605     0.610     0.605      1267


Training XGBoost with MI features (selected inside each CV fold)...


Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 6, 'clf__n_estimators': 200}

XGBoost
[[207 346]
 [140 574]]
              precision    recall  f1-score   support

        fake      0.597     0.374     0.460       553
        real      0.624     0.804     0.703       714

    accuracy                          0.616      1267
   macro avg      0.610     0.589     0.581      1267
weighted avg      0.612     0.616     0.597      1267



Results to table, merged with the v2 baseline + Dummy results (I-4 exit criterion)

In [11]:
def results_to_dataframe(results_dict, method_name):
    rows = []
    for model_name, data in results_dict.items():
        test_m = data["test"]
        rows.append(
            {
                "Pipeline": "Proposed",
                "Method": method_name,
                "Model": model_name,
                "Test Accuracy": test_m["accuracy"],
                "Test Macro-F1": test_m["macro_f1"],
                "Test Fake Precision": test_m["fake_precision"],
                "Test Fake Recall": test_m["fake_recall"],
                "Test Fake F1": test_m["fake_f1"],
                "Test Real F1": test_m["real_f1"],
                "Test Confusion Matrix": test_m["confusion_matrix"],
                "Best Params": data["best_params"],
            }
        )
    return pd.DataFrame(rows)


df_chi2 = results_to_dataframe(results_chi2, "Chi-square")
df_mi = results_to_dataframe(results_mi, "Mutual Information")
proposed_results = pd.concat([df_chi2, df_mi], ignore_index=True)

baseline_results = pd.read_csv("baseline_results_v2.csv")

final_results = pd.concat([baseline_results, proposed_results], ignore_index=True)
final_results = final_results.sort_values("Test Macro-F1", ascending=False)
final_results

,Pipeline,Method,Model,Test Accuracy,Test Macro-F1,Test Fake Precision,Test Fake Recall,Test Fake F1,Test Real F1,Test Confusion Matrix,Best Params
13,Proposed,Mutual Information,Logistic Regression,0.623520,0.620338,0.563545,0.609403,0.585578,0.655098,"[[337, 216], [261, 453]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced..."
8,Proposed,Chi-square,Logistic Regression,0.617206,0.614709,0.555556,0.614828,0.583691,0.645727,"[[340, 213], [272, 442]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced..."
9,Proposed,Chi-square,SVM,0.616417,0.612646,0.556876,0.593128,0.574431,0.650862,"[[328, 225], [261, 453]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced'}"
4,Baseline,TF-IDF only,Logistic Regression (balanced),0.613260,0.607687,0.555950,0.566004,0.560932,0.654443,"[[313, 240], [250, 464]]",NaN
11,Proposed,Chi-square,Random Forest,0.622731,0.606542,0.582057,0.481013,0.526733,0.686352,"[[266, 287], [191, 523]]","{'clf__max_depth': None, 'clf__min_samples_spl..."
3,Baseline,TF-IDF only,Logistic Regression,0.627466,0.606418,0.596200,0.453888,0.515400,0.697436,"[[251, 302], [170, 544]]",NaN
14,Proposed,Mutual Information,SVM,0.606946,0.602157,0.547826,0.569620,0.558511,0.645804,"[[315, 238], [260, 454]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced'}"
2,Baseline,TF-IDF only,Naive Bayes,0.626677,0.598796,0.605263,0.415913,0.493033,0.704560,"[[230, 323], [150, 564]]",NaN
5,Baseline,TF-IDF only,SVM,0.604578,0.596059,0.549057,0.526221,0.537396,0.654721,"[[291, 262], [239, 475]]",NaN
16,Proposed,Mutual Information,Random Forest,0.610103,0.595213,0.562633,0.479204,0.517578,0.672848,"[[265, 288], [206, 508]]","{'clf__max_depth': None, 'clf__min_samples_spl..."


In [12]:
final_results.to_csv("model_comparison_results_v2.csv", index=False)